In [49]:
import spacy
import json
import random

nlp = spacy.load("en_core_web_sm") ##You get POS tags, dependency parse, noun_chunks → ABSA ready instead of doing them manually.

In [53]:
sentences_data = [
    "The food was delicious but the service was slow.",
    "I loved the cozy atmosphere and friendly staff.",
    "Prices were a bit high compared to other restaurants.",
    "Was in NOLA the last day of Mardi Gras through Friday.  The jambalaya is the best I've ever had!" 
    "The pasta lacked flavor, though the portion was generous.",
    "Overall, I would recommend this place for family dinners."
]

In [43]:
for s in sentences_data:
    print(f"\nSentence: {s}")
    doc = nlp(s)
    print("Tokens:", [t.text for t in doc])
    print("POS Tags:", [(t.text, t.pos_) for t in doc])
    print("Noun Chunks:", [chunk.text for chunk in doc.noun_chunks])

    print("Dependencies:", [(t.text, t.dep_, t.head.text) for t in doc])


Sentence: The food was delicious but the service was slow.
Tokens: ['The', 'food', 'was', 'delicious', 'but', 'the', 'service', 'was', 'slow', '.']
POS Tags: [('The', 'DET'), ('food', 'NOUN'), ('was', 'AUX'), ('delicious', 'ADJ'), ('but', 'CCONJ'), ('the', 'DET'), ('service', 'NOUN'), ('was', 'AUX'), ('slow', 'ADJ'), ('.', 'PUNCT')]
Noun Chunks: ['The food', 'the service']
Dependencies: [('The', 'det', 'food'), ('food', 'nsubj', 'was'), ('was', 'ROOT', 'was'), ('delicious', 'acomp', 'was'), ('but', 'cc', 'was'), ('the', 'det', 'service'), ('service', 'nsubj', 'was'), ('was', 'conj', 'was'), ('slow', 'acomp', 'was'), ('.', 'punct', 'was')]

Sentence: I loved the cozy atmosphere and friendly staff.
Tokens: ['I', 'loved', 'the', 'cozy', 'atmosphere', 'and', 'friendly', 'staff', '.']
POS Tags: [('I', 'PRON'), ('loved', 'VERB'), ('the', 'DET'), ('cozy', 'ADJ'), ('atmosphere', 'NOUN'), ('and', 'CCONJ'), ('friendly', 'ADJ'), ('staff', 'NOUN'), ('.', 'PUNCT')]
Noun Chunks: ['I', 'the cozy atm

In [44]:

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

def sentiment_scores(data):
    sid_obj = SentimentIntensityAnalyzer()
    compound_scores = []
    for text in data:
        sentiment_dict = sid_obj.polarity_scores(text)
        compound_scores.append(sentiment_dict['compound'])

        print(f"Sentence: {text}")
        print(f"Sentiment Scores: {sentiment_dict}")
        print(f"Negative Sentiment: {sentiment_dict['neg']*100:.2f}%")
        print(f"Neutral Sentiment: {sentiment_dict['neu']*100:.2f}%")
        print(f"Positive Sentiment: {sentiment_dict['pos']*100:.2f}%\n")

    # Overall sentiment: average of compound scores
    avg_compound = sum(compound_scores)/len(compound_scores)
    if avg_compound >= 0.05:
        overall = "Positive"
    elif avg_compound <= -0.05:
        overall = "Negative"
    else:
        overall = "Neutral"

    print(f"Overall Sentiment: {overall}")

In [45]:
sentiment_scores(sentences_data)

Sentence: The food was delicious but the service was slow.
Sentiment Scores: {'neg': 0.0, 'neu': 0.773, 'pos': 0.227, 'compound': 0.3291}
Negative Sentiment: 0.00%
Neutral Sentiment: 77.30%
Positive Sentiment: 22.70%

Sentence: I loved the cozy atmosphere and friendly staff.
Sentiment Scores: {'neg': 0.0, 'neu': 0.458, 'pos': 0.542, 'compound': 0.7964}
Negative Sentiment: 0.00%
Neutral Sentiment: 45.80%
Positive Sentiment: 54.20%

Sentence: Prices were a bit high compared to other restaurants.
Sentiment Scores: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Negative Sentiment: 0.00%
Neutral Sentiment: 100.00%
Positive Sentiment: 0.00%

Sentence: The pasta lacked flavor, though the portion was generous.
Sentiment Scores: {'neg': 0.0, 'neu': 0.708, 'pos': 0.292, 'compound': 0.5106}
Negative Sentiment: 0.00%
Neutral Sentiment: 70.80%
Positive Sentiment: 29.20%

Sentence: Overall, I would recommend this place for family dinners.
Sentiment Scores: {'neg': 0.0, 'neu': 0.762, 'pos': 0.

In [48]:
sid= SentimentIntensityAnalyzer()

def extract_aspects(doc_test):
    return [chunk.root.text.lower() for chunk in doc_test.noun_chunks]


def find_opinions(doc_values, aspects_data):
    opinions = []
    for token in doc_values:
        # adjective modifying noun (e.g., "delicious food")
        if token.dep_ == "amod" and token.head.text.lower() in aspects_data:
            opinions.append((token.head.text.lower(), token.text.lower(), token))
        # adjective complement (e.g., "food was delicious")
        elif token.dep_ == "acomp":
            for child in token.head.children:
                if child.dep_ == "nsubj" and child.text.lower() in aspects_data:
                    opinions.append((child.text.lower(), token.text.lower(), token))
        # adverb modifying adjective/verb (e.g., "bit high")
        elif token.dep_ == "advmod" and token.head.pos_ in ["ADJ", "VERB"]:
            for child in token.head.children:
                if child.dep_ == "nsubj" and child.text.lower() in aspects_data:
                    opinions.append((child.text.lower(), token.text.lower(), token))
    return opinions


def score_opinions(opinions):
    aspect_sentiments = {}

    for aspect, opinion_word, token in opinions:
        base_score = sid.polarity_scores(opinion_word)['compound']

        # --- Negation ---
        neg_factor = -1 if any(child.dep_ == "neg" for child in token.children) else 1
        if token.head:
            neg_factor *= -1 if any(child.dep_ == "neg" for child in token.head.children) else 1

        # --- Intensifiers ---
        intensity_factor = 1.0
        for child in token.children:
            if child.dep_ == "advmod" and child.text.lower() in ["very", "extremely", "highly", "so", "really"]:
                intensity_factor += 0.5
        if token.head:
            for child in token.head.children:
                if child.dep_ == "advmod" and child.text.lower() in ["very", "extremely", "highly", "so", "really"]:
                    intensity_factor += 0.5

        # Adjusted score
        final_score = base_score * neg_factor * intensity_factor
        if aspect not in aspect_sentiments:
            aspect_sentiments[aspect] = []
        aspect_sentiments[aspect].append(final_score)

    # Aggregate
    for aspect, scores in aspect_sentiments.items():
        avg = sum(scores) / len(scores)
        sentiment = "Positive" if avg >= 0.05 else "Negative" if avg <= -0.05 else "Neutral"
        aspect_sentiments[aspect] = (sentiment, avg)

    return aspect_sentiments


def analyze_aspect_sentiment(sentences):
    results = {}
    for s in sentences:
        doc = nlp(s)
        aspects = extract_aspects(doc)
        opinions = find_opinions(doc, aspects)
        aspect_sentiments = score_opinions(opinions)
        results[s] = aspect_sentiments
    return results





Sentence: The food was delicious but the service was slow.
  Aspect: food            | Sentiment: Positive | Score: 0.57
  Aspect: service         | Sentiment: Neutral  | Score: 0.00

Sentence: I loved the cozy atmosphere and friendly staff.
  Aspect: atmosphere      | Sentiment: Neutral  | Score: 0.00
  Aspect: staff           | Sentiment: Positive | Score: 0.49

Sentence: Prices were a bit high compared to other restaurants.
  Aspect: prices          | Sentiment: Neutral  | Score: 0.00
  Aspect: restaurants     | Sentiment: Neutral  | Score: 0.00

Sentence: The pasta lacked flavor, though the portion was generous.
  Aspect: portion         | Sentiment: Positive | Score: 0.51

Sentence: Overall, I would recommend this place for family dinners.
  Aspect: i               | Sentiment: Neutral  | Score: 0.00


In [54]:
for s in sentences_data:
    doc_data = nlp(s)
    aspect = extract_aspects(doc_data)
    opinion = find_opinions(doc_data, aspect)
    print(f"Aspects: {aspect}")
    print(f"Opinions: {opinion}")

Aspects: ['food', 'service']
Opinions: [('food', 'delicious', delicious), ('service', 'slow', slow)]
Aspects: ['i', 'atmosphere', 'staff']
Opinions: [('atmosphere', 'cozy', cozy), ('staff', 'friendly', friendly)]
Aspects: ['prices', 'restaurants']
Opinions: [('prices', 'high', high), ('restaurants', 'other', other)]
Aspects: ['nola', 'gras', 'friday', 'jambalaya', 'i', 'pasta', 'flavor', 'portion']
Opinions: [('pasta', 'ever', ever), ('portion', 'generous', generous)]
Aspects: ['i', 'place', 'dinners']
Opinions: [('i', 'overall', Overall)]


In [ ]:
analysis = analyze_aspect_sentiment(sentences_data)

for sentence, aspects in analysis.items():
    print(f"\nSentence: {sentence}")
    for aspect, (sentiment, score) in aspects.items():
        print(f"  Aspect: {aspect:15} | Sentiment: {sentiment:8} | Score: {score:.2f}")

In [50]:
sample_reviews = []
with open("../data/yelp_sample_20k.json", "r", encoding="utf-8") as f:
    all_lines = f.readlines()
    sample_lines = random.sample(all_lines, 5)  # pick 5 random reviews
    for line in sample_lines:
        data = json.loads(line)
        sample_reviews.append(data["text"])  # assuming Yelp JSON has "text" field
        
print("Sample reviews:\n")
for r in sample_reviews:
    print("-", r[:120].replace("\n", " "), "...")

Sample reviews:

- I've tried most of the famous cheesesteaks in Philly. This one is good, but not the best one I've tried. I got the chees ...
- I've only had Jamaican food twice. My first experience was here. I had curry goat and fried plantains. I enjoyed my meal ...
- Kim is the beat. Food Delicious.  My granddaughter loved her dinner as I did too.  I had a great Eggplant Parm Sandwich. ...
- Casino is so gross. Their is no ventilation at all, it is hot and very smokey inside. This place was very depressing to  ...
- It's $5 for all you can eat, it's not going to be the best of the best, but it is good. I love going there and they help ...


In [51]:
analysis = analyze_aspect_sentiment(sample_reviews)

for sentence, aspects in analysis.items():
    print(f"\nSentence: {sentence}")
    for aspect, (sentiment, score) in aspects.items():
        print(f"  Aspect: {aspect:15} | Sentiment: {sentiment:8} | Score: {score:.2f}")


Sentence: I've tried most of the famous cheesesteaks in Philly. This one is good, but not the best one I've tried. I got the cheesesteak with peppers and onions and provolone cheese. I like that it wasnt very greasy and it was a good quality of meat. However, it wasnt the most flavorful cheesesteak i've ever had.
Its definitely good- just not the best.
  Aspect: cheesesteaks    | Sentiment: Neutral  | Score: 0.00
  Aspect: one             | Sentiment: Negative | Score: -0.10
  Aspect: cheese          | Sentiment: Neutral  | Score: 0.00
  Aspect: quality         | Sentiment: Positive | Score: 0.44
  Aspect: cheesesteak     | Sentiment: Neutral  | Score: 0.00
  Aspect: i               | Sentiment: Neutral  | Score: 0.00

Sentence: I've only had Jamaican food twice. My first experience was here. I had curry goat and fried plantains. I enjoyed my meal. While service was very slow, I didn't mind it as the restaurant was small but busy. Unfortunately, my second Jamaican food experience (in 

In [52]:
doc = nlp("Was in NOLA the last day of Mardi Gras through Friday.  The jambalaya is the best I've ever had!")
for token in doc:
    print(token.text, "→", token.dep_, "→", token.head.text)

Was → ROOT → Was
in → prep → Was
NOLA → pobj → in
the → det → day
last → amod → day
day → npadvmod → Was
of → prep → day
Mardi → compound → Gras
Gras → pobj → of
through → prep → Was
Friday → pobj → through
. → punct → Was
  → dep → .
The → det → jambalaya
jambalaya → nsubj → is
is → ROOT → is
the → det → best
best → attr → is
I → nsubj → had
've → aux → had
ever → advmod → had
had → relcl → best
! → punct → is
